# 01 · Dataset inspection and audit

**Goal:** know the data before modelling. This notebook runs on a CPU runtime and needs the dataset
(restored from the archive by the setup cell).

It covers: split sizes, image integrity, formats / sizes / colour modes, image-to-mask pairing, visual inspection,
defect sizes, a resolution check, and a test of the loading code.

## What the audit found (outputs are cleared in this copy; re-run to reproduce)

- **Splits:** 1,548 normal training images; 800 test images (178 normal, 622 defective); 30 defect types in total.
  Training contains normal images only (enforced in code, see `get_split`).
- **Integrity:** 2,970 image and mask files inspected, 0 unreadable, all PNG.
- **Colour:** screw and zipper are grayscale in both train and test; the other five categories are RGB.
- **Sizes:** bottle 900², capsule 1000², carpet / screw / toothbrush / zipper 1024², metal_nut 700² (all square).
- **Pairing:** all 622 defective test images have a mask of identical size; none missing, none unmatched.
- **Defect size** (fraction of the image): screw medians are about 0.25 to 0.5%, capsule crack / faulty imprint /
  poke about 0.2 to 0.3%; metal_nut `flip` covers about 48% because the whole object is flipped.

## Preprocessing decisions that follow (implemented in `src/data/dataset.py`)

- Resize the **whole image** to a square (no centre crop, so no ground-truth defect pixels are discarded), using `INTER_AREA`.
- Replicate grayscale to 3 channels; normalise with ImageNet mean and std (the backbone was pretrained with it).
- Resize masks with `INTER_AREA` and threshold at 0.5; metrics are computed at the resized resolution.
- No training augmentation (a flipped metal_nut is itself a defect class).

In [ ]:
import logging, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

NEED_DATA = True   # True: this notebook reads the images

drive.mount("/content/drive")

REPO_URL = "https://github.com/tsegaeph/industrial-anomaly-detection.git"
REPO_DIR = Path("/content/repo")
DRIVE_PROJECT = Path("/content/drive/MyDrive/industrial-anomaly-detection")
DRIVE_DATA = DRIVE_PROJECT / "datasets" / "mvtec_anomaly_detection"
ARCHIVE = DRIVE_PROJECT / "datasets" / "mvtec_7cat.tar"
DATA_ROOT = Path("/content/industrial-anomaly-detection/datasets/mvtec_anomaly_detection")
EXP = DRIVE_PROJECT / "experiments"
REPORTS = DRIVE_PROJECT / "reports"
CATS = ("bottle", "capsule", "carpet", "metal_nut", "screw", "toothbrush", "zipper")


def sh(*cmd) -> None:
    """Run a command, print its output, raise on failure."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode:
        raise RuntimeError(f"{cmd[0]} failed: {r.stderr.strip()}")


# Code always comes from GitHub. After a pull that changes modules you already imported,
# use Runtime -> Restart session (this keeps /content, so the dataset stays).
if REPO_DIR.exists():
    sh("git", "-C", REPO_DIR, "pull")
else:
    sh("git", "clone", REPO_URL, REPO_DIR)
sys.path.insert(0, str(REPO_DIR))


def dataset_ok() -> bool:
    return all((DATA_ROOT / c / "train" / "good").is_dir() for c in CATS)


def ensure_dataset() -> None:
    """Restore the dataset on Colab's local disk if it is missing or incomplete."""
    if dataset_ok():
        print("Dataset already complete on local disk.")
        return
    shutil.rmtree(DATA_ROOT.parent.parent, ignore_errors=True)          # remove any partial copy
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if ARCHIVE.exists():                       # one large file: much faster than thousands of small ones
        sh("cp", ARCHIVE, "/content/mvtec.tar")
        sh("tar", "-xf", "/content/mvtec.tar", "-C", DATA_ROOT.parent)
        Path("/content/mvtec.tar").unlink(missing_ok=True)
    if not dataset_ok():                       # no archive, or extraction failed
        print("Archive unavailable or incomplete, copying file by file (slow)...")
        shutil.rmtree(DATA_ROOT.parent.parent, ignore_errors=True)
        DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(DRIVE_DATA, DATA_ROOT)
    assert dataset_ok(), "Dataset incomplete after copy"
    print("Dataset ready.")


if NEED_DATA:
    ensure_dataset()

logging.basicConfig(level=logging.INFO)

In [ ]:
import pandas as pd
from src.data.mvtec import CATEGORIES

REPORT_DIR = REPORTS / "dataset_audit"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATASET = DATA_ROOT
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

## 1. Split sizes and defect types

In [ ]:
def count_images(directory: Path) -> int:
    if not directory.exists():
        return 0
    return sum(1 for f in directory.rglob("*") if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS)


category_rows, defect_rows = [], []
for category in CATEGORIES:
    category_path = LOCAL_DATASET / category
    train_good = count_images(category_path / "train" / "good")      # training: normal only
    test_good = count_images(category_path / "test" / "good")
    test_path = category_path / "test"
    defect_types = sorted(f.name for f in test_path.iterdir() if f.is_dir() and f.name != "good")

    test_defective = 0
    for defect_type in defect_types:
        n_test = count_images(test_path / defect_type)
        n_mask = count_images(category_path / "ground_truth" / defect_type)
        test_defective += n_test
        defect_rows.append({"Category": category, "Defect Type": defect_type,
                            "Test Images": n_test, "Ground Truth Masks": n_mask})
    category_rows.append({"Category": category, "Train Normal": train_good, "Test Normal": test_good,
                          "Test Defective": test_defective, "Total Test": test_good + test_defective})

category_df, defect_df = pd.DataFrame(category_rows), pd.DataFrame(defect_rows)
print("CATEGORY-LEVEL SUMMARY")
display(category_df)
print("\nDEFECT-TYPE BREAKDOWN")
display(defect_df)
print("\nTotals | train normal:", category_df["Train Normal"].sum(),
      "| test normal:", category_df["Test Normal"].sum(),
      "| test defective:", category_df["Test Defective"].sum(),
      "| masks:", defect_df["Ground Truth Masks"].sum())

## 2. Image integrity, formats, sizes and colour modes

Questions: are all files readable, what formats are present, what are the dimensions and colour modes?

In [ ]:
from PIL import Image

metadata, corrupted_files = [], []


def inspect_image(path: Path):
    """Read image metadata and verify that the file is readable."""
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:            # reopen: verify() invalidates the object
            return {"width": img.width, "height": img.height, "mode": img.mode, "format": img.format}
    except Exception as error:
        corrupted_files.append({"File": str(path), "Error": str(error)})
        return None


for category in CATEGORIES:
    for split in ["train", "test", "ground_truth"]:
        for path in (LOCAL_DATASET / category / split).rglob("*"):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            info = inspect_image(path)
            if info is not None:
                metadata.append({"Category": category, "Split": split, "File": path.name,
                                 "Width": info["width"], "Height": info["height"],
                                 "Mode": info["mode"], "Format": info["format"], "Path": str(path)})

metadata_df = pd.DataFrame(metadata)
print("Inspected:", len(metadata_df), "| corrupted/unreadable:", len(corrupted_files))
print("\nFORMATS");     display(metadata_df["Format"].value_counts().rename_axis("Format").reset_index(name="Count"))
print("\nCOLOUR MODES"); display(metadata_df["Mode"].value_counts().rename_axis("Mode").reset_index(name="Count"))
print("\nDIMENSIONS BY CATEGORY AND SPLIT")
display(metadata_df.groupby(["Category", "Split"]).agg(
    Images=("File", "count"),
    Widths=("Width", lambda x: sorted(x.unique())),
    Heights=("Height", lambda x: sorted(x.unique())),
    Modes=("Mode", lambda x: sorted(x.unique()))).reset_index())
if corrupted_files:
    display(pd.DataFrame(corrupted_files))

## 3. Image-to-mask pairing

In [ ]:
pairing_results = []
for category in CATEGORIES:
    test_path = LOCAL_DATASET / category / "test"
    mask_path = LOCAL_DATASET / category / "ground_truth"

    for defect_dir in sorted(test_path.iterdir()):
        if not defect_dir.is_dir() or defect_dir.name == "good":
            continue
        mask_dir = mask_path / defect_dir.name
        test_images = [p for p in defect_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        masks = ([p for p in mask_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
                 if mask_dir.exists() else [])

        image_by_stem = {p.stem: p for p in test_images}
        mask_by_stem = {p.stem.removesuffix("_mask"): p for p in masks}     # MVTec masks end in "_mask"
        image_names, mask_names = set(image_by_stem), set(mask_by_stem)

        mismatches = 0
        for stem in image_names & mask_names:
            with Image.open(image_by_stem[stem]) as im, Image.open(mask_by_stem[stem]) as m:
                mismatches += int(im.size != m.size)

        pairing_results.append({"Category": category, "Defect Type": defect_dir.name,
                                "Test Images": len(test_images), "Masks": len(masks),
                                "Matched": len(image_names & mask_names),
                                "Missing Masks": len(image_names - mask_names),
                                "Unmatched Masks": len(mask_names - image_names),
                                "Dimension Mismatches": mismatches})

pairing_df = pd.DataFrame(pairing_results)
display(pairing_df)
print("Matched pairs:", pairing_df["Matched"].sum(), "| missing masks:", pairing_df["Missing Masks"].sum(),
      "| unmatched masks:", pairing_df["Unmatched Masks"].sum(),
      "| dimension mismatches:", pairing_df["Dimension Mismatches"].sum())
assert pairing_df["Missing Masks"].sum() == 0
assert pairing_df["Unmatched Masks"].sum() == 0
assert pairing_df["Dimension Mismatches"].sum() == 0
print("All image-mask pairing checks passed.")

## 4. Visual inspection and defect sizes

`build_index` scans the folder tree once and returns one row per image. It raises if a defective image has no mask
or if any training image is not `good`. `plot_category` shows one normal image plus two defect types (image, mask,
overlay) per category, and `defect_area_stats` measures how much of the image each defect covers.
Figures and `defect_area_stats.csv` are saved to `reports/dataset_audit/` on Drive.

In [ ]:
from src.data.mvtec import build_index
from src.visualization.inspection import plot_category, defect_area_stats

df = build_index(DATA_ROOT)
assert len(df) == 1548 + 800, len(df)
print(df.groupby(["category", "split", "label"]).size().unstack(fill_value=0))

for c in CATEGORIES:
    plot_category(df, c, save_dir=REPORT_DIR)

stats = defect_area_stats(df)
stats.to_csv(REPORT_DIR / "defect_area_stats.csv", index=False)
with pd.option_context("display.max_rows", 100, "display.float_format", "{:.4%}".format):
    print(stats)

## 5. Resolution check

Shows the smallest defect of a category at several input sizes (same physical window), with the number of defect
pixels that survive. It motivated 256 px as the main resolution and a 384 px sensitivity experiment.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from src.data.mvtec import load_image, load_mask


def smallest_defect(df, category):
    """Return (area_fraction, row, image, mask) for the smallest-defect test image."""
    best = None
    for r in df[(df.category == category) & (df.label == 1)].itertuples():
        img = load_image(r.image_path)
        m = load_mask(r.mask_path, img.shape[:2])
        if best is None or m.mean() < best[0]:
            best = (m.mean(), r, img, m)
    return best


def show_resolutions(df, category, resolutions=(224, 256, 384, 512), win_frac=0.15):
    frac, r, img, mask = smallest_defect(df, category)
    ys, xs = np.nonzero(mask)
    cy, cx = ys.mean() / mask.shape[0], xs.mean() / mask.shape[1]          # centroid, relative
    fig, axes = plt.subplots(1, len(resolutions) + 1, figsize=(3 * (len(resolutions) + 1), 3.4))
    axes[0].imshow(img)
    axes[0].set_title(f"{category} {r.defect_type}\narea {frac:.3%}", fontsize=9)
    axes[0].add_patch(plt.Rectangle((cx * img.shape[1] - img.shape[1] * win_frac / 2,
                                     cy * img.shape[0] - img.shape[0] * win_frac / 2),
                                    img.shape[1] * win_frac, img.shape[0] * win_frac, fill=False, ec="yellow", lw=2))
    for ax, res in zip(axes[1:], resolutions):
        im_r = cv2.resize(img, (res, res), interpolation=cv2.INTER_AREA)
        m_r = cv2.resize(mask.astype(np.uint8), (res, res), interpolation=cv2.INTER_NEAREST) > 0
        half = int(res * win_frac / 2)
        y0, x0 = max(int(cy * res) - half, 0), max(int(cx * res) - half, 0)
        crop = im_r[y0:y0 + 2 * half, x0:x0 + 2 * half]
        mcrop = m_r[y0:y0 + 2 * half, x0:x0 + 2 * half]
        ax.imshow(crop, interpolation="nearest")
        ax.contour(mcrop, levels=[0.5], colors="yellow", linewidths=1)
        ax.set_title(f"{res}px  defect={int(m_r.sum())} px", fontsize=9)
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


for c in ("screw", "capsule", "carpet"):
    show_resolutions(df, c)

## 6. Sanity check of the data loader

Checks batch shape and dtype, that training batches contain no defect pixels and only label 0, and that every
defective test image keeps a non-zero mask after resizing to 256 px (a zero would mean the resize erased a defect).
Normalised means are not 0: screw has a bright background and metal_nut a dark one, which is expected.
Defective test images are sorted by defect type, so a metal_nut batch starts with `bent`, not `flip`.

In [ ]:
import torch
from src.data.dataset import get_split, make_loader

for cat in ("screw", "metal_nut"):
    train_df, test_df = get_split(df, cat, "train"), get_split(df, cat, "test")

    batch = next(iter(make_loader(train_df, image_size=256, batch_size=8)))
    x = batch["image"]
    print(f"{cat} train batch:", tuple(x.shape), x.dtype, f"mean={x.mean():.2f} std={x.std():.2f}",
          "| mask sum:", batch["mask"].sum().item(), "| labels:", batch["label"].tolist())

    defective = test_df[test_df.label == 1]
    b = next(iter(make_loader(defective, image_size=256, batch_size=8)))
    print(f"{cat} defective test, mask pixels per image:", b["mask"].sum(dim=(1, 2, 3)).int().tolist())